# 🧬 Self-Replicating Agent — Kaggle GPU (Option B)

**Model:** Qwen2.5-Coder-14B-Instruct from **Kaggle Model Hub** (pre-cached, no download)  
**GPU:** T4 16 GB — model loaded with 4-bit quant (~7 GB VRAM)  
**Shim:** Tiny FastAPI server that mimics Ollama's API — zero changes to agent code

### Before running — required setup:
1. `Settings → Accelerator → GPU T4 x1` ✅
2. `Settings → Internet → On` ✅
3. Add the Qwen2.5-Coder model: `+ Add Input → Models → search "qwen2.5-coder" → 14B-Instruct` ✅

### Optional — add Kaggle Secrets for cloud API fallback:
- `GROQ_API_KEY`, `CEREBRAS_API_KEY` (both free)

In [ ]:
# ── Cell 1: Verify GPU ─────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
gpu_info = r.stdout.strip()
print('GPU:', gpu_info or '❌ NOT FOUND — enable GPU T4 in Settings!')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Find Qwen2.5-Coder-14B model path ─────────────────────────────
import os, glob

# Kaggle attaches models under /kaggle/input/
# Search for the config.json which marks the model root
candidates = glob.glob('/kaggle/input/**/config.json', recursive=True)
qwen_paths = [
    os.path.dirname(p) for p in candidates
    if 'qwen' in p.lower() or 'coder' in p.lower()
]

if not qwen_paths:
    # Show all available inputs to help debug
    print('❌ Qwen model not found. Available inputs:')
    for d in os.listdir('/kaggle/input'):
        print(f'  /kaggle/input/{d}/')
    print('\n👉 Add the model: + Add Input → Models → search qwen2.5-coder → 14b-instruct')
    MODEL_PATH = None
else:
    MODEL_PATH = qwen_paths[0]
    print(f'✅ Found model at: {MODEL_PATH}')
    print('Contents:', os.listdir(MODEL_PATH)[:8])

print(f'\nMODEL_PATH = {MODEL_PATH!r}')

In [ ]:
# ── Cell 3: Install dependencies ───────────────────────────────────────────
import subprocess, sys

pkgs = [
    'fastapi',
    'uvicorn[standard]',
    'bitsandbytes',          # 4-bit quantization
    'accelerate',            # device_map="cuda" support
    'langchain-groq',
    'langchain-core',
    'langchain-openai',
    'langgraph',
    'langchain-community',
    'langchain-google-genai',
    'cerebras-cloud-sdk',
    'openai',
]

print('Installing dependencies (1-2 min)...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs,
    capture_output=True, text=True
)
print('STDERR (tail):', r.stderr[-300:] if r.stderr else 'none')
print('✅ Done' if r.returncode == 0 else f'❌ pip exit {r.returncode}')

In [ ]:
# ── Cell 4: Write the LLM shim server ──────────────────────────────────────
# This FastAPI app mimics Ollama's API so llm_client.py needs zero changes.

SHIM_CODE = '''
import json, logging, time, uuid, os
from typing import List, Optional
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
log = logging.getLogger("shim")

app = FastAPI()
_model = None
_tokenizer = None
_model_name = os.environ.get("SHIM_MODEL_NAME", "qwen2.5-coder:14b")

def load_model():
    global _model, _tokenizer
    path = os.environ["SHIM_MODEL_PATH"]
    log.info(f"Loading tokenizer from {path}")
    _tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    log.info("Loading model with 4-bit quantization...")
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    _model = AutoModelForCausalLM.from_pretrained(
        path, quantization_config=quant, device_map="cuda",
        trust_remote_code=True, torch_dtype=torch.float16,
    )
    _model.eval()
    gb = torch.cuda.memory_allocated() / 1e9
    log.info(f"Model ready — {gb:.1f} GB VRAM used")

class Msg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: str = "qwen2.5-coder:14b"
    messages: List[Msg]
    max_tokens: Optional[int] = 4096
    temperature: Optional[float] = 0.2
    stream: Optional[bool] = False

@app.get("/api/tags")
def tags():
    return {"models": [{"name": _model_name, "size": 9_000_000_000}]}

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": _model is not None}

@app.post("/v1/chat/completions")
def chat(req: ChatReq):
    if _model is None:
        return JSONResponse({"error": "Model not loaded"}, status_code=503)
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    text = _tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
    prompt_len = inputs["input_ids"].shape[1]
    t0 = time.time()
    with torch.no_grad():
        out = _model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 4096,
            temperature=max(req.temperature or 0.2, 1e-6),
            do_sample=(req.temperature or 0.2) > 0.01,
            pad_token_id=_tokenizer.eos_token_id,
            eos_token_id=_tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    new_ids = out[0][prompt_len:]
    response = _tokenizer.decode(new_ids, skip_special_tokens=True)
    n = len(new_ids)
    log.info(f"{n} tokens in {elapsed:.1f}s ({n/elapsed:.1f} tok/s)")
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": _model_name,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": response}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": prompt_len, "completion_tokens": n, "total_tokens": prompt_len + n},
    }

load_model()
uvicorn.run(app, host="0.0.0.0", port=11434, log_level="warning")
'''

with open('/kaggle/working/llm_shim.py', 'w') as f:
    f.write(SHIM_CODE)
print('✅ Shim server written to /kaggle/working/llm_shim.py')

In [ ]:
# ── Cell 5: Load model + start shim server ─────────────────────────────────
# Model loading takes ~60-90s on T4. Server starts in background thread.

import subprocess, sys, os, time, urllib.request, json

assert MODEL_PATH, "Run Cell 2 first — model path not found!"

env = os.environ.copy()
env['SHIM_MODEL_PATH'] = MODEL_PATH
env['SHIM_MODEL_NAME'] = 'qwen2.5-coder:14b'

print(f'Starting LLM shim server (loading {MODEL_PATH})...')
print('This takes ~60-90s for model loading — watch for "Model ready" below...')

shim_proc = subprocess.Popen(
    [sys.executable, '/kaggle/working/llm_shim.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream logs until server is ready
import threading
ready = threading.Event()

def log_reader():
    for line in shim_proc.stdout:
        print('[shim]', line.rstrip())
        if 'Model ready' in line:
            ready.set()

t = threading.Thread(target=log_reader, daemon=True)
t.start()

# Wait up to 3 min for model to load
if ready.wait(timeout=180):
    print('\n✅ Model loaded!')
else:
    print('\n⚠️ Still loading — waiting a bit more...')
    time.sleep(30)

# Verify server is responding
for attempt in range(10):
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            data = json.load(r)
            models = [m['name'] for m in data.get('models', [])]
            print(f'✅ Shim server ready — models: {models}')
            break
    except Exception as e:
        print(f'Waiting for server... ({attempt+1}/10)')
        time.sleep(5)
else:
    print('❌ Server not responding after 50s')

In [ ]:
# ── Cell 6: Quick smoke test ───────────────────────────────────────────────
import urllib.request, json, time

print('Testing model via shim...')
t0 = time.time()

payload = json.dumps({
    'model': 'qwen2.5-coder:14b',
    'messages': [
        {'role': 'system', 'content': 'You are a Python expert.'},
        {'role': 'user',   'content': 'Write a one-liner to flatten a nested list. Answer in one line only.'}
    ],
    'max_tokens': 80,
    'temperature': 0.1,
}).encode()

req = urllib.request.Request(
    'http://localhost:11434/v1/chat/completions',
    data=payload,
    headers={'Content-Type': 'application/json'},
    method='POST'
)
with urllib.request.urlopen(req, timeout=120) as r:
    resp = json.load(r)

elapsed = time.time() - t0
content = resp['choices'][0]['message']['content']
n_tokens = resp['usage']['completion_tokens']
tok_s = n_tokens / elapsed

print(f'Response: {content.strip()}')
print(f'Speed: {tok_s:.1f} tok/s ({elapsed:.1f}s)')
print(f'GPU check: {"✅ GPU (>10 tok/s)" if tok_s > 10 else "⚠️ Slow — check GPU is enabled"}')

In [ ]:
# ── Cell 7: Clone evolution code from GitHub ───────────────────────────────
import subprocess, os

REPO   = 'https://github.com/balaji33k/SelfReplicatingAgent.git'
BRANCH = 'fresh-main'
DEST   = '/kaggle/working/SelfReplicatingAgent'

# Try GitHub token from Kaggle secrets (needed if repo is private)
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    REPO = REPO.replace('https://', f'https://{token}@')
    print('Using GitHub token')
except Exception:
    print('No GITHUB_TOKEN — trying as public repo')

if os.path.exists(DEST):
    r = subprocess.run(['git', '-C', DEST, 'pull'], capture_output=True, text=True)
    print('Updated repo:', r.stdout.strip())
else:
    r = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, DEST],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f'✅ Cloned to {DEST}')
    else:
        print('❌ Clone failed:', r.stderr)
        raise RuntimeError('Cannot clone repo')

files = os.listdir(f'{DEST}/generations/gen_1/')
print(f'gen_1 files ({len(files)}):', sorted(files)[:10])

In [ ]:
# ── Cell 8: Configure environment ─────────────────────────────────────────
import os, json

DEST = '/kaggle/working/SelfReplicatingAgent'

# Point to our shim (it looks like Ollama to llm_client.py)
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL']    = 'qwen2.5-coder:14b'

# Load optional cloud API keys (fallback chain)
try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for key in ['GROQ_API_KEY', 'CEREBRAS_API_KEY', 'SAMBANOVA_API_KEY', 'GoogleAPIKey']:
        try:
            val = sc.get_secret(key)
            if val:
                os.environ[key] = val
                print(f'✅ {key} loaded')
        except Exception:
            pass
except Exception:
    pass

# Write user_config.json
os.makedirs(f'{DEST}/data', exist_ok=True)
with open(f'{DEST}/data/user_config.json', 'w') as f:
    json.dump({'model': 'qwen2.5-coder:14b'}, f)

print('\nConfig:')
print(f'  OLLAMA_BASE_URL = {os.environ["OLLAMA_BASE_URL"]}')
print(f'  OLLAMA_MODEL    = {os.environ["OLLAMA_MODEL"]}')

In [ ]:
# ── Cell 9: Run Evolution Gen 1 ────────────────────────────────────────────
# 10 xarray tasks → skip instantly
# 10 LCB tasks × ~5-8 min each on T4 = ~1-1.5 hrs total

import subprocess, sys, os

DEST    = '/kaggle/working/SelfReplicatingAgent'
GEN_DIR = f'{DEST}/generations/gen_1'

env = os.environ.copy()
env['PYTHONPATH'] = GEN_DIR

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--generation', '1'],
    cwd=GEN_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print('🚀 Evolution running — live output:')
print('=' * 60)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    print('\n⚠️ Interrupted')
    proc.kill()

proc.wait()
print('=' * 60)
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 10: Results summary ───────────────────────────────────────────────
import json, os

DEST = '/kaggle/working/SelfReplicatingAgent'
pf   = f'{DEST}/data/gen_1_progress.json'

if not os.path.exists(pf):
    print('No results yet'); raise SystemExit

with open(pf) as f:
    data = json.load(f)

results   = data.get('results', {})
task_list = data.get('task_list', [])
success   = [t for t, r in results.items() if r['status'] == 'success']
failed    = [t for t, r in results.items() if r['status'] == 'fail' and r.get('error_type') != 'SkippedUnsolvable']
skipped   = [t for t, r in results.items() if r.get('error_type') == 'SkippedUnsolvable']

pct = len(success) / max(len(task_list) - len(skipped), 1) * 100
print(f'=== Gen 1 Results ===')
print(f'Progress : {len(results)}/{len(task_list)}')
print(f'✅ Success : {len(success)}  ({pct:.0f}% pass rate on attempted tasks)')
print(f'❌ Failed  : {len(failed)}')
print(f'⏭️  Skipped : {len(skipped)}')

if failed:
    print('\nFailed tasks:')
    for t in failed[:5]:
        r = results[t]
        print(f'  {t}: {r.get("error_type","")} — {str(r.get("stderr",""))[:100]}')

In [ ]:
# ── Cell 11: Save results to Kaggle output ─────────────────────────────────
import shutil, os

DEST = '/kaggle/working/SelfReplicatingAgent'
OUT  = '/kaggle/working/evolution_results'
os.makedirs(OUT, exist_ok=True)

for src, dst in [
    (f'{DEST}/data/gen_1_progress.json', f'{OUT}/gen_1_progress.json'),
    (f'{DEST}/data/evolution_log.json',  f'{OUT}/evolution_log.json'),
    (f'{DEST}/generations/gen_1/generation.log', f'{OUT}/generation_1.log'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved: {os.path.basename(dst)}')

gen2 = f'{DEST}/generations/gen_2'
if os.path.exists(gen2):
    shutil.copytree(gen2, f'{OUT}/gen_2', dirs_exist_ok=True)
    print('Saved: gen_2/')

print(f'\nAll saved to {OUT} — visible in Kaggle Output tab')